機械学習を使ってフェイクニュースかどうかを判別する<br>
使用したデータセット
https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets?resource=download <br>
各ツールをインストール -> pip install pandas scikit-learn matplot

In [1]:
#FakeとTrueのデータセットを結合して、FakeNewsdataset.csvという新しいファイルを作成するコード
import pandas as pd

fake = pd.read_csv('../Data/Fake.csv')
fake['label'] = 'fake' #fakeデータにラベルを追加

true = pd.read_csv('../Data/True.csv')
true['label'] = 'true' #trueデータにラベルを追加

df = pd.concat([fake, true], ignore_index=True) #fakeとtrueのデータを結合してDataFrameに変換
df.to_csv('../Data/FakeNewsdataset.csv', index=False) #csvファイルをDataディレクトリに保存

結合したデータセットを読み込んで、学習用データとテスト用データに分割

In [2]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("../Data/FakeNewsdataset.csv", quotechar='"', sep=",") #結合したデータセットを読み込む
texts = df["text"] #テキスト列を抽出
label = df["label"] #ラベル列を抽出

x_train, x_test, y_train, y_test = train_test_split(texts, label, test_size=0.2, random_state=2) #データセットを訓練用とテスト用に分割

print(x_train.head()) #訓練用の先頭データを表示
print(y_train.head())

10185    MSNBC s Brian Williams spoke the truth on Tues...
17269    Don t worry the liberal elite always know what...
29275     (In this Jan. 31 story, in 11th paragraph cor...
12668    Every single day for months on end, new eviden...
39427    DUBLIN (Reuters) - The continuing failure of I...
Name: text, dtype: str
10185    fake
17269    fake
29275    true
12668    fake
39427    true
Name: label, dtype: str


テキストデータの前処理をそれぞれ実行<br>
nltkをコマンドでインストール -> pip install nltk<br>
nltk.download("stopwords")を一回実行

In [3]:
import re
from nltk.corpus import stopwords


def NormalizeText(text):
    text = text.lower() #テキストを小文字変換
    text = text.replace('\n', ' ') #改行をスペースに置換
    text = re.sub(r'[^\w\s]', '', text) #特殊文字を削除
    text = str.strip(text) #テキストの前後の空白を削除
    return text

def TokenizeText(text):
    tokens = text.split() #テキストをスペースで分割してトークン化
    return tokens

def StopWordRemoval(tokens):
    stop_words = set(stopwords.words('english')) #英語のストップワードを取得
    filtered_tokens = [word for word in tokens if word not in stop_words] #ストップワードを除去
    return filtered_tokens

x_train_Normalized = x_train.apply(NormalizeText) #訓練用テキストを正規化
x_train_Tokenized = x_train_Normalized.apply(TokenizeText) #訓練用テキストをトークン化
x_train_Filtered = x_train_Tokenized.apply(StopWordRemoval) #ストップワードを除去

x_test_Normalized = x_test.apply(NormalizeText) #テスト用も同様に処理
x_test_Tokenized = x_test_Normalized.apply(TokenizeText)
x_test_Filtered = x_test_Tokenized.apply(StopWordRemoval)

x_train_Filtered.head() #処理された訓練用テキストの先頭データを表示


10185    [msnbc, brian, williams, spoke, truth, tuesday...
17269    [worry, liberal, elite, always, know, best, li...
29275    [jan, 31, story, 11th, paragraph, corrects, sh...
12668    [every, single, day, months, end, new, evidenc...
39427    [dublin, reuters, continuing, failure, irish, ...
Name: text, dtype: object

labelのデータをfake,trueから0,1に変換

In [4]:
def map_label(label):
    if label == 'fake':
        return 0
    else:
        return 1

y_train_mapped = y_train.apply(map_label) #ラベルを数値に変換

y_test_mapped = y_test.apply(map_label) #テスト用のラベルも同様に変換

y_train_mapped.head() #変換された訓練用ラベルの先頭データを表示

10185    0
17269    0
29275    1
12668    0
39427    1
Name: label, dtype: int64